# Práctica · Spark DataFrames · Clientes y pedidos

Completa las celdas indicadas. Trabaja sobre el entorno dockerizado incluido en `spark_jupyter/`.

In [2]:
from iniciar_spark import get_spark
from pyspark.sql import functions as F

spark = get_spark("PracticaClientesPedidos")
print("SparkSession creada")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 16:01:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada
Versión de Spark: 4.1.1


In [3]:
from pathlib import Path

data_dir = Path("/opt/spark-apps/datos")
clientes_path = data_dir / "clientes.csv"
pedidos_path = data_dir / "pedidos.csv"

print("Ruta clientes:", clientes_path)
print("Ruta pedidos:", pedidos_path)


Ruta clientes: /opt/spark-apps/datos/clientes.csv
Ruta pedidos: /opt/spark-apps/datos/pedidos.csv


## 1. Lee ambos CSV y muestra su esquema y algunas filas.

In [4]:
# TODO
df_clientes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv(str(clientes_path))
 
df_pedidos = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ";") \
    .csv(str(pedidos_path))
 
# Mostrar esquemas para validar que las columnas están separadas correctamente [cite: 101, 106]
print("--- Esquema de Clientes ---")
df_clientes.printSchema()
 
print("--- Esquema de Pedidos ---")
df_pedidos.printSchema()
 
# Mostrar contenido inicial [cite: 101]
print("--- Muestra de Clientes ---")
df_clientes.show(5)
 
print("--- Muestra de Pedidos ---")
df_pedidos.show(5)

--- Esquema de Clientes ---
root
 |-- id_cliente: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- segmento: string (nullable = true)

--- Esquema de Pedidos ---
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: double (nullable = true)
 |-- precio_unitario: integer (nullable = true)

--- Muestra de Clientes ---
+----------+------+----------+--------+
|id_cliente|nombre|    ciudad|segmento|
+----------+------+----------+--------+
|         1|   Ana|  Sevilla |Estandar|
|         2|  Luis|    Bilbao| Premium|
|         3| Marta| Alicante |Estandar|
|         4| Pablo|   Madrid | Premium|
|         5| Lucia|    Bilbao| Premium|
+----------+------+----------+--------+
only showing top 5 rows
--- Muestra de Pedidos ---
+---------+----------+----------+---------+--------+---------------+
|id_pedido|id_clie

## 2. Limpia el DataFrame de clientes: trim en ciudad y dropDuplicates.

In [5]:
# TODO
# Eliminar duplicados para asegurar que cada cliente sea único [cite: 12, 48, 103]

df_clientes_limpio = df_clientes.dropDuplicates()
 
# Aplicar trim a la columna 'ciudad' para eliminar espacios en blanco accidentales 

# También es buena práctica limpiar la columna 'nombre' si fuera necesario [cite: 104]

df_clientes_limpio = df_clientes_limpio.withColumn("ciudad", F.trim(F.col("ciudad")))
 
# Gestión de nulos (opcional pero recomendado según el enunciado) [cite: 14, 49, 105]

# Por ejemplo, rellenar ciudades desconocidas si las hubiera

df_clientes_limpio = df_clientes_limpio.na.fill({"ciudad": "Desconocida"})
 
# Mostrar resultados de la limpieza [cite: 143]

print(f"Registros originales: {df_clientes.count()}")

print(f"Registros tras eliminar duplicados: {df_clientes_limpio.count()}")
 
df_clientes_limpio.show(5)
 

Registros originales: 43
Registros tras eliminar duplicados: 40
+----------+------+--------+--------+
|id_cliente|nombre|  ciudad|segmento|
+----------+------+--------+--------+
|        18| Jorge| Sevilla| Premium|
|        40| Tomas|Valencia|Estandar|
|         5| Lucia|  Bilbao| Premium|
|         6|Carlos|Alicante|Estandar|
|        14|Sergio|  Murcia|Estandar|
+----------+------+--------+--------+
only showing top 5 rows


## 3. Convierte tipos en pedidos y crea la columna `importe`.

In [6]:
# TODO
from pyspark.sql import functions as F

# 1. Convertir tipos: nos aseguramos de que cantidad sea entero y precio sea decimal (double) 
df_pedidos_trans = df_pedidos.withColumn("cantidad", F.col("cantidad").cast("int")) \
                             .withColumn("precio_unitario", F.col("precio_unitario").cast("double"))

# 2. Crear la columna importe = cantidad * precio_unitario 
df_pedidos_trans = df_pedidos_trans.withColumn("importe", F.col("cantidad") * F.col("precio_unitario"))

# 3. Validar tipos de datos y mostrar resultado [cite: 106, 142]
print("--- Esquema de Pedidos Transformado ---")
df_pedidos_trans.printSchema()

print("--- Muestra de Pedidos con Importe ---")
df_pedidos_trans.show(5)

--- Esquema de Pedidos Transformado ---
root
 |-- id_pedido: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- importe: double (nullable = true)

--- Muestra de Pedidos con Importe ---
+---------+----------+----------+---------+--------+---------------+-------+
|id_pedido|id_cliente|     fecha| producto|cantidad|precio_unitario|importe|
+---------+----------+----------+---------+--------+---------------+-------+
|     1001|        12|2025-03-05|    Ratón|       6|           16.0|   96.0|
|     1002|        32|2025-02-02|    Ratón|       3|           19.0|   57.0|
|     1003|         5|2025-03-07|  Monitor|       2|          210.0|  420.0|
|     1004|        13|2025-03-18|Impresora|       4|          205.0|  820.0|
|     1005|        41|2025-03-10|  Teclado|       5|           40.0|  200.0|
+---------+-

## 4. Haz un join entre clientes y pedidos.

In [7]:
# TODO
# Realizar el join (Inner Join por defecto usando 'id_cliente')
df_join = df_clientes_limpio.join(df_pedidos_trans, on="id_cliente", how="inner")

# Analizar qué registros se pierden y por qué
conteo_pedidos = df_pedidos_trans.count()
conteo_join = df_join.count()

print(f"Pedidos originales: {conteo_pedidos}")
print(f"Pedidos tras el join: {conteo_join}")
print(f"Registros perdidos: {conteo_pedidos - conteo_join}")
print("Explicación: Los registros se pierden si hay pedidos con un 'id_cliente' que no existe en la tabla de clientes (registros huérfanos) o si se eliminaron clientes en la fase de limpieza.")

df_join.show(5)

Pedidos originales: 120
Pedidos tras el join: 110
Registros perdidos: 10
Explicación: Los registros se pierden si hay pedidos con un 'id_cliente' que no existe en la tabla de clientes (registros huérfanos) o si se eliminaron clientes en la fase de limpieza.
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|id_cliente|nombre|  ciudad|segmento|id_pedido|     fecha| producto|cantidad|precio_unitario|importe|
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|        12|Miguel|Zaragoza|Estandar|     1001|2025-03-05|    Ratón|       6|           16.0|   96.0|
|        32|Andres|Zaragoza| Premium|     1002|2025-02-02|    Ratón|       3|           19.0|   57.0|
|         5| Lucia|  Bilbao| Premium|     1003|2025-03-07|  Monitor|       2|          210.0|  420.0|
|        13| Irene|  Madrid| Premium|     1004|2025-03-18|Impresora|       4|          205.0|  820.0|
|         5| Lucia|  Bilbao|

## 5. Filtra ventas Premium con importe >= 100.

In [8]:
# TODO
# Filtrar ventas donde el importe es mayor o igual a 100 y el segmento es Premium [cite: 115]
df_premium = df_join.filter((F.col("importe") >= 100) & (F.col("segmento") == "Premium"))

print("--- Ventas Segmento Premium (Importe >= 100) ---")
df_premium.show(5)

--- Ventas Segmento Premium (Importe >= 100) ---


+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|id_cliente|nombre|  ciudad|segmento|id_pedido|     fecha| producto|cantidad|precio_unitario|importe|
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
|         5| Lucia|  Bilbao| Premium|     1003|2025-03-07|  Monitor|       2|          210.0|  420.0|
|        13| Irene|  Madrid| Premium|     1004|2025-03-18|Impresora|       4|          205.0|  820.0|
|        37| Oscar|Alicante| Premium|     1013|2025-02-16| Portátil|       5|          741.0| 3705.0|
|        16| Diego| Granada| Premium|     1014|2025-02-17|   Webcam|       2|           78.0|  156.0|
|         4| Pablo|  Madrid| Premium|     1018|2025-02-06|   Webcam|       3|           53.0|  159.0|
+----------+------+--------+--------+---------+----------+---------+--------+---------------+-------+
only showing top 5 rows


## 6. Clasifica pedidos con `when` en Alto / Medio / Bajo.

In [9]:
# TODO
# Clasificar los pedidos usando F.when 
df_clasificado = df_join.withColumn(
    "categoria_pedido",
    F.when(F.col("importe") > 500, "Alto")
     .when(F.col("importe") >= 100, "Medio")
     .otherwise("Bajo")
)

print("--- Pedidos Clasificados ---")
df_clasificado.select("id_pedido", "importe", "categoria_pedido").show(5)

--- Pedidos Clasificados ---
+---------+-------+----------------+
|id_pedido|importe|categoria_pedido|
+---------+-------+----------------+
|     1088|  124.0|           Medio|
|     1024|  252.0|           Medio|
|     1009|   38.0|            Bajo|
|     1094|  768.0|            Alto|
|     1056|  108.0|           Medio|
+---------+-------+----------------+
only showing top 5 rows


## 7. Calcula agregaciones por ciudad y segmento.

In [10]:
# TODO
# Calcular número de pedidos, ingresos totales y medias agrupando por ciudad y segmento 
df_agregaciones = df_join.groupBy("ciudad", "segmento").agg(
    F.count("id_pedido").alias("numero_pedidos"),
    F.sum("importe").alias("ingresos_totales"),
    F.round(F.avg("importe"), 2).alias("media_importe")
)

print("--- Agregaciones por Ciudad y Segmento ---")
df_agregaciones.orderBy(F.col("ingresos_totales").desc()).show()

--- Agregaciones por Ciudad y Segmento ---
+----------+--------+--------------+----------------+-------------+
|    ciudad|segmento|numero_pedidos|ingresos_totales|media_importe|
+----------+--------+--------------+----------------+-------------+
|    Bilbao|Estandar|            19|         11685.0|        615.0|
|   Granada| Premium|             7|          9161.0|      1308.71|
|  Alicante|Estandar|            10|          9113.0|        911.3|
|  Alicante| Premium|             7|          7472.0|      1067.43|
|    Bilbao| Premium|             8|          5674.0|       810.57|
|   Sevilla|Estandar|             6|          3091.0|       515.17|
|    Madrid|Estandar|             5|          2719.0|        543.8|
|  Zaragoza|Estandar|             6|          2710.0|       451.67|
|    Madrid| Premium|             5|          2347.0|        469.4|
|    Murcia|Estandar|             8|          2058.0|        294.0|
|  Valencia|Estandar|             6|          1593.0|        265.5|
|    

## 8. Crea una vista temporal y haz una consulta SQL.

In [11]:
# TODO
# Crear una vista temporal en memoria [cite: 119]
df_join.createOrReplaceTempView("vista_ventas")

# Realizar una consulta SQL significativa (Ej: Top 3 clientes con mayor gasto) [cite: 120]
df_sql = spark.sql("""
    SELECT id_cliente, nombre, SUM(importe) as gasto_total, COUNT(id_pedido) as total_pedidos
    FROM vista_ventas
    GROUP BY id_cliente, nombre
    ORDER BY gasto_total DESC
    LIMIT 3
""")

print("--- Top 3 Clientes por Gasto (Vía SQL) ---")
df_sql.show()

--- Top 3 Clientes por Gasto (Vía SQL) ---


[Stage 51:>                                                         (0 + 1) / 1]

+----------+------+-----------+-------------+
|id_cliente|nombre|gasto_total|total_pedidos|
+----------+------+-----------+-------------+
|        22| Ruben|     7112.0|            3|
|        37| Oscar|     6758.0|            5|
|        10| David|     6697.0|            5|
+----------+------+-----------+-------------+



## 9. Usa `sample()` y `randomSplit()`.

In [ ]:
# TODO

## 10. Guarda el resultado en Parquet en `/opt/spark-apps/salida/resultado_parquet` y léelo de nuevo.

In [ ]:
# TODO

## 11. Responde brevemente en Markdown:
- ¿Qué ventaja tiene usar join?
- ¿Qué diferencia hay entre sample y randomSplit?
- ¿Qué pedido se pierde en el inner join y por qué?

## 12. Cierra la sesión de Spark.

In [ ]:
spark.stop()